# LoLdle Classic — evaluación de seguimiento de restricciones

Notebook para evaluar un modelo local servido por LM Studio en el modo Classic de LoLdle. Se ejecuta una sola corrida por modelo: todos los objetivos de la corrida usan la misma semilla y cada partida reinicia la conversación.

Flujo:

1. Edita la configuración de la siguiente celda.
2. Ejecuta las celdas hasta el self-check.
3. Inicia LM Studio con su servidor OpenAI-compatible activo.
4. Define MODEL_NAME y cambia RUN_NOW a True en la última celda.
5. Ejecuta la última celda para jugar y ver el reporte en vivo.

No se guardan archivos de resultados: la traza queda en la variable results de la sesión.

In [ ]:
from pathlib import Path
import csv
import json
import random
import re
import unicodedata
from html import escape
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None


# Hiperparámetros de la corrida
MAX_ATTEMPTS = 10
NUM_GAMES = 1
SEED = 777
TEMPERATURE = 0
MODEL_NAME = "qwen3.5-4b-instruct-revised"
CHAMPIONS_FILE_PATH = Path("data/champions_173_prompt.txt")

# Conexión y controles técnicos (no tienes que poner /completions al final de la URL, se agrega automáticamente)
BASE_URL = "http://localhost:1234/v1"
API_KEY = "lm-studio"
MAX_TOKENS = 64
TIMEOUT_SECONDS = 300
PROMPT_ROLE = "system"  # usa "user" si la plantilla del modelo ignora system
RUN_NOW = False


def resolve_data_path(value):
    path = Path(value).expanduser()
    candidates = [path] if path.is_absolute() else [
        Path.cwd() / path,
        Path.cwd() / "loldle" / path,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"No se encontró el roster en {value!s}. "
        "Ejecuta el notebook desde loldle/ o ajusta CHAMPIONS_FILE_PATH."
    )


DATA_PATH = resolve_data_path(CHAMPIONS_FILE_PATH)
print(f"Roster: {DATA_PATH}")

## 1. Datos de campeones y prompt

El archivo se valida contra su encabezado real. Los campos positions, species y regions son multivalor; gender, resource y range_type son escalares; release_year es entero.

In [ ]:
EXPECTED_HEADER = (
    "name", "gender", "positions", "species", "resource",
    "range_type", "regions", "release_year",
)
SET_FIELDS = ("positions", "species", "regions")
SCALAR_FIELDS = ("gender", "resource", "range_type")
FEEDBACK_FIELDS = (
    "gender", "positions", "species", "resource",
    "range_type", "regions", "release_year",
)


def split_values(value):
    return frozenset(part.strip() for part in value.split(",") if part.strip())


def load_roster(path):
    with path.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream, delimiter="|")
        if tuple(reader.fieldnames or ()) != EXPECTED_HEADER:
            raise ValueError(
                f"Encabezado inesperado: {reader.fieldnames!r}; "
                f"se esperaba {EXPECTED_HEADER!r}"
            )
        champions = {}
        for line_number, row in enumerate(reader, start=2):
            if any(row.get(field) is None for field in EXPECTED_HEADER):
                raise ValueError(f"Fila {line_number} incompleta en {path.name}")
            values = {field: row[field].strip() for field in EXPECTED_HEADER}
            if any(not value for value in values.values()):
                raise ValueError(f"Fila {line_number} vacía en {path.name}")
            name = values["name"]
            if name in champions:
                raise ValueError(f"Campeón duplicado: {name}")
            try:
                release_year = int(values["release_year"])
            except ValueError as exc:
                raise ValueError(
                    f"Año inválido para {name}: {values['release_year']}"
                ) from exc
            champions[name] = {
                "name": name,
                "gender": values["gender"],
                "positions": split_values(values["positions"]),
                "species": split_values(values["species"]),
                "resource": values["resource"],
                "range_type": values["range_type"],
                "regions": split_values(values["regions"]),
                "release_year": release_year,
            }
    if len(champions) != 173:
        raise ValueError(f"Se esperaban 173 campeones, se encontraron {len(champions)}")
    return champions


def fold_text(value):
    normalized = unicodedata.normalize("NFKD", str(value))
    return "".join(
        char for char in normalized if not unicodedata.combining(char)
    ).casefold()


def canonical_key(value):
    return "".join(char for char in fold_text(value) if char.isalnum())


CHAMPIONS = load_roster(DATA_PATH)
NAMES = tuple(CHAMPIONS)
ROSTER_TEXT = DATA_PATH.read_text(encoding="utf-8")

# El patrón permite variantes de mayúsculas, tildes, espacios y apóstrofes.
NAME_PATTERNS = []
for champion_name in NAMES:
    tokens = re.findall(r"[a-z0-9]+", fold_text(champion_name))
    pattern = re.compile(
        r"(?<![a-z0-9])"
        + r"[^a-z0-9]*".join(re.escape(token) for token in tokens)
        + r"(?![a-z0-9])"
    )
    NAME_PATTERNS.append((champion_name, pattern))
NAME_PATTERNS.sort(key=lambda item: len(canonical_key(item[0])), reverse=True)


SYSTEM_PROMPT = (
    "<|no_reasoning|>\n"
    "Eres un jugador de LoLdle Classic. Debes adivinar el campeón secreto de "
    "League of Legends usando solamente el roster y el feedback de la partida.\n\n"
    "En cada intento recibirás siete atributos: género, posiciones, especie, "
    "recurso, alcance, región y año de lanzamiento. VERDE significa coincidencia "
    "exacta. AMARILLO significa coincidencia parcial en un atributo multivalor. "
    "GRIS significa que no hay coincidencia. En el año, ↑ significa que el "
    "objetivo fue lanzado después del año propuesto y ↓ significa que fue lanzado "
    "antes.\n\n"
    "Debes conservar todas las restricciones reveladas, no repetir campeones y "
    "no proponer nombres incompatibles con el feedback anterior. Cada respuesta "
    "debe ser únicamente el nombre exacto de un campeón que aparezca en el roster: "
    "sin explicación, sin comillas, sin JSON, sin viñetas y sin texto adicional.\n\n"
    "ROSTER COMPLETO (campos separados por |):\n"
    + ROSTER_TEXT
)


def parse_response(raw):
    result = {"name": None, "format_ok": False, "reason": ""}
    if not isinstance(raw, str) or not raw.strip():
        result["reason"] = "La respuesta está vacía o no es texto."
        return result
    folded = fold_text(raw.strip())
    found = [
        name for name, pattern in NAME_PATTERNS
        if pattern.search(folded)
    ]
    unique_found = list(dict.fromkeys(found))
    if len(unique_found) == 0:
        result["reason"] = "No se reconoció ningún campeón del roster."
        return result
    if len(unique_found) > 1:
        result["reason"] = (
            "La respuesta menciona más de un campeón: "
            + ", ".join(unique_found)
        )
        return result
    name = unique_found[0]
    pattern = next(pattern for candidate, pattern in NAME_PATTERNS if candidate == name)
    result["name"] = name
    result["format_ok"] = pattern.fullmatch(folded) is not None
    if not result["format_ok"]:
        result["reason"] = "Se reconoció el campeón, pero había texto o puntuación adicional."
    return result

print(f"Campeones cargados: {len(CHAMPIONS)}")

## 2. Feedback y restricciones

El checker marca como contradicción un verde incompatible, un valor prohibido por gris o un año fuera de los límites indicados por flechas. Los amarillos se muestran, pero no crean restricciones duras.

In [ ]:
LABELS = {
    "gender": "Género",
    "positions": "Posiciones",
    "species": "Especie",
    "resource": "Recurso",
    "range_type": "Alcance",
    "regions": "Regiones",
    "release_year": "Año",
}
STATUS_LABELS = {
    "green": "VERDE · coincidencia exacta",
    "yellow": "AMARILLO · coincidencia parcial",
    "gray": "GRIS · sin coincidencia",
    "higher": "↑ objetivo posterior",
    "lower": "↓ objetivo anterior",
}
STATUS_SYMBOLS = {
    "green": "✓",
    "yellow": "≈",
    "gray": "×",
    "higher": "↑",
    "lower": "↓",
}
STATUS_COLORS = {
    "green": "#16803c",
    "yellow": "#b7791f",
    "gray": "#52525b",
    "higher": "#2563eb",
    "lower": "#7c3aed",
}


def score(guess_name, target_name):
    guess = CHAMPIONS[guess_name]
    target = CHAMPIONS[target_name]
    feedback = {}
    for field in FEEDBACK_FIELDS:
        if field == "release_year":
            guess_year = guess[field]
            target_year = target[field]
            feedback[field] = (
                "green" if guess_year == target_year
                else "higher" if target_year > guess_year
                else "lower"
            )
        elif field in SET_FIELDS:
            overlap = guess[field] & target[field]
            feedback[field] = (
                "green" if guess[field] == target[field]
                else "yellow" if overlap
                else "gray"
            )
        else:
            feedback[field] = "green" if guess[field] == target[field] else "gray"
    return feedback


def is_solved(feedback):
    return bool(feedback) and all(status == "green" for status in feedback.values())


def empty_constraints():
    constraints = {
        field: {"required": None, "forbidden": set()}
        for field in (*SCALAR_FIELDS, *SET_FIELDS)
    }
    constraints["release_year"] = {
        "exact": None,
        "min_exclusive": None,
        "max_exclusive": None,
    }
    return constraints


def update_constraints(constraints, guess_name, feedback):
    guess = CHAMPIONS[guess_name]
    for field in (*SCALAR_FIELDS, *SET_FIELDS):
        status = feedback[field]
        if status == "green":
            constraints[field]["required"] = guess[field]
        elif status == "gray":
            if field in SET_FIELDS:
                constraints[field]["forbidden"].update(guess[field])
            else:
                constraints[field]["forbidden"].add(guess[field])
    year = guess["release_year"]
    if feedback["release_year"] == "green":
        constraints["release_year"]["exact"] = year
    elif feedback["release_year"] == "higher":
        current = constraints["release_year"]["min_exclusive"]
        constraints["release_year"]["min_exclusive"] = (
            year if current is None else max(current, year)
        )
    elif feedback["release_year"] == "lower":
        current = constraints["release_year"]["max_exclusive"]
        constraints["release_year"]["max_exclusive"] = (
            year if current is None else min(current, year)
        )


def _value_text(value):
    if isinstance(value, (set, frozenset)):
        return ", ".join(sorted(value))
    return str(value)


def constraint_violations(guess_name, constraints):
    guess = CHAMPIONS[guess_name]
    violations = []
    for field in (*SCALAR_FIELDS, *SET_FIELDS):
        required = constraints[field]["required"]
        if required is not None and guess[field] != required:
            violations.append({
                "attribute": field,
                "value": _value_text(guess[field]),
                "rule": f"debía coincidir con {_value_text(required)}",
            })
        forbidden = constraints[field]["forbidden"]
        if field in SET_FIELDS:
            hits = set(guess[field]) & forbidden
            if hits:
                violations.append({
                    "attribute": field,
                    "value": ", ".join(sorted(hits)),
                    "rule": "valores descartados por feedback gris",
                })
        elif guess[field] in forbidden:
            violations.append({
                "attribute": field,
                "value": str(guess[field]),
                "rule": "valor descartado por feedback gris",
            })

    year = guess["release_year"]
    year_constraints = constraints["release_year"]
    exact = year_constraints["exact"]
    if exact is not None and year != exact:
        violations.append({
            "attribute": "release_year",
            "value": str(year),
            "rule": f"el año exacto revelado era {exact}",
        })
    elif (
        year_constraints["min_exclusive"] is not None
        and year <= year_constraints["min_exclusive"]
    ):
        violations.append({
            "attribute": "release_year",
            "value": str(year),
            "rule": f"el objetivo debía ser posterior a {year_constraints['min_exclusive']}",
        })
    elif (
        year_constraints["max_exclusive"] is not None
        and year >= year_constraints["max_exclusive"]
    ):
        violations.append({
            "attribute": "release_year",
            "value": str(year),
            "rule": f"el objetivo debía ser anterior a {year_constraints['max_exclusive']}",
        })
    return violations


def evaluate_attempt(raw, target_name, history, constraints):
    parsed = parse_response(raw)
    eligible = any(event.get("feedback") is not None for event in history)
    event = {
        "raw_response": raw if isinstance(raw, str) else repr(raw),
        "guess": parsed["name"],
        "format_ok": parsed["format_ok"],
        "format_violation": not parsed["format_ok"],
        "invalid": parsed["name"] is None,
        "eligible": eligible,
        "repeated": False,
        "constraint_violation": False,
        "violations": [],
        "feedback": None,
        "solved": False,
        "reason": parsed["reason"],
    }
    if event["invalid"]:
        event["status"] = "invalid"
        return event

    event["repeated"] = event["guess"] in {
        item.get("guess") for item in history if item.get("guess")
    }
    event["violations"] = constraint_violations(event["guess"], constraints)
    event["constraint_violation"] = bool(event["violations"])
    event["feedback"] = score(event["guess"], target_name)
    event["solved"] = is_solved(event["feedback"])
    update_constraints(constraints, event["guess"], event["feedback"])

    if event["solved"]:
        event["status"] = "solved"
    elif event["constraint_violation"]:
        event["status"] = "constraint_violation"
    elif event["repeated"]:
        event["status"] = "repeated"
    else:
        event["status"] = "valid"
    return event


def feedback_message(event):
    if event["invalid"]:
        return (
            "La respuesta anterior fue inválida: "
            + (event["reason"] or "no se reconoció un único campeón")
            + f" Intenta de nuevo; este fue el intento {event['turn']} de {MAX_ATTEMPTS}. "
            "Responde únicamente con un nombre del roster."
        )

    lines = []
    champion = CHAMPIONS[event["guess"]]
    if event["format_violation"]:
        lines.append("FORMATO INCORRECTO: se detectó texto o puntuación adicional.")
    if event["repeated"]:
        lines.append("REPETICIÓN: ya habías propuesto este campeón.")
    if event["constraint_violation"]:
        details = "; ".join(
            f"{LABELS[item['attribute']]} ({item['rule']})"
            for item in event["violations"]
        )
        lines.append("VIOLACIÓN DE RESTRICCIÓN: " + details)
    lines.append("Feedback del intento " + str(event["turn"]) + ":")
    for field in FEEDBACK_FIELDS:
        value = _value_text(champion[field])
        status = event["feedback"][field]
        if field == "release_year":
            if status == "green":
                description = f"VERDE: año exacto {value}"
            elif status == "higher":
                description = f"↑ el objetivo es posterior a {value}"
            else:
                description = f"↓ el objetivo es anterior a {value}"
        else:
            description = STATUS_LABELS[status]
        lines.append(f"- {LABELS[field]} = {value} -> {description}")
    lines.append(
        f"Elige el siguiente campeón en el intento {event['turn'] + 1} de {MAX_ATTEMPTS}. "
        "Usa todo el feedback y responde solo con el nombre."
    )
    return "\n".join(lines)

## 3. Cliente OpenAI-compatible de LM Studio y conversación

In [ ]:
def _http_json(path, body):
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    request = Request(
        BASE_URL.rstrip("/") + path,
        data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    try:
        with urlopen(request, timeout=TIMEOUT_SECONDS) as response:
            payload = json.load(response)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")[:500]
        raise RuntimeError(
            f"LM Studio respondió HTTP {exc.code} en {path}: {detail}"
        ) from exc
    except (URLError, TimeoutError) as exc:
        raise RuntimeError(
            f"No se pudo conectar con LM Studio en {BASE_URL}: {exc}"
        ) from exc
    except (json.JSONDecodeError, ValueError) as exc:
        raise RuntimeError(
            f"LM Studio devolvió JSON inválido en {path}"
        ) from exc
    if not isinstance(payload, dict):
        raise RuntimeError(f"LM Studio devolvió un objeto inválido en {path}")
    return payload


def query_model(messages):
    if not isinstance(MODEL_NAME, str) or not MODEL_NAME.strip():
        raise ValueError("Define MODEL_NAME antes de consultar LM Studio.")
    payload = _http_json("/chat/completions", {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "stream": False,
    })
    choices = payload.get("choices")
    if not isinstance(choices, list) or not choices:
        raise RuntimeError("La respuesta no contiene choices.")
    if not isinstance(choices[0], dict):
        raise RuntimeError("La primera choice no es un objeto.")
    message = choices[0].get("message")
    if not isinstance(message, dict) or not isinstance(message.get("content"), str):
        raise RuntimeError("La respuesta no contiene message.content de texto.")
    content = message["content"]
    if not content.strip():
        finish_reason = choices[0].get("finish_reason")
        if message.get("reasoning_content"):
            raise RuntimeError(
                "LM Studio agotó la generación en reasoning_content y no devolvió una respuesta final "
                f"(finish_reason={finish_reason!r})."
            )
        raise RuntimeError(
            f"LM Studio devolvió una respuesta final vacía (finish_reason={finish_reason!r})."
        )
    return content


def build_messages(history):
    initial = (
        f"Comienza la partida. Este es el intento 1 de {MAX_ATTEMPTS}. "
        "Responde solo con el nombre de un campeón del roster."
    )
    if PROMPT_ROLE == "system":
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": initial},
        ]
    elif PROMPT_ROLE == "user":
        messages = [
            {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + initial},
        ]
    else:
        raise ValueError("PROMPT_ROLE debe ser 'system' o 'user'.")

    for event in history:
        messages.append({"role": "assistant", "content": event["raw_response"]})
        messages.append({"role": "user", "content": feedback_message(event)})
    return messages

## 4. Loop del juego y vista en vivo

In [ ]:
def show_event(event, game_number):
    label = f"Partida {game_number} · intento {event['turn']}"
    if event["invalid"]:
        print(f"{label}: INVÁLIDO — {event['reason']}")
        print(f"  Respuesta cruda: {event['raw_response']!r}")
        return

    flags = []
    if event["format_violation"]:
        flags.append("formato")
    if event["repeated"]:
        flags.append("repetido")
    if event["constraint_violation"]:
        flags.append("violación")
    suffix = f" [{', '.join(flags)}]" if flags else ""
    print(f"{label}: {event['guess']}{suffix}")
    print(f"  Respuesta cruda: {event['raw_response']!r}")

    if display is None:
        for field in FEEDBACK_FIELDS:
            value = _value_text(CHAMPIONS[event["guess"]][field])
            print(f"  {LABELS[field]}: {value} -> {event['feedback'][field]}")
        return

    cells = []
    champion = CHAMPIONS[event["guess"]]
    for field in FEEDBACK_FIELDS:
        status = event["feedback"][field]
        value = _value_text(champion[field])
        cells.append(
            "<td style='background:{};color:white;padding:8px;text-align:center'>"
            "<b>{}</b><br>{}<br><small>{}</small></td>".format(
                STATUS_COLORS[status],
                escape(LABELS[field]),
                escape(value),
                escape(STATUS_LABELS[status]),
            )
        )
    notices = []
    if event["format_violation"]:
        notices.append("formato fuera de lo exigido")
    if event["repeated"]:
        notices.append("repetición")
    if event["constraint_violation"]:
        notices.append("violación: " + "; ".join(
            f"{LABELS[item['attribute']]} — {item['rule']}"
            for item in event["violations"]
        ))
    notice_html = escape(" · ".join(notices)) if notices else "sin incumplimientos detectados"
    display(HTML(
        "<div style='border:1px solid #d4d4d8;border-radius:8px;padding:10px;margin:8px 0'>"
        "<b>{}</b><br><table style='width:100%;border-spacing:3px'><tr>{}</tr></table>"
        "<small>{}</small></div>".format(
            escape(label),
            "".join(cells),
            notice_html,
        )
    ))


def choose_targets():
    if not isinstance(NUM_GAMES, int) or isinstance(NUM_GAMES, bool):
        raise ValueError("NUM_GAMES debe ser entero.")
    if not 1 <= NUM_GAMES <= len(NAMES):
        raise ValueError(f"NUM_GAMES debe estar entre 1 y {len(NAMES)}.")
    return random.Random(SEED).sample(list(NAMES), NUM_GAMES)


def validate_config():
    if not isinstance(MAX_ATTEMPTS, int) or isinstance(MAX_ATTEMPTS, bool) or MAX_ATTEMPTS < 1:
        raise ValueError("MAX_ATTEMPTS debe ser entero positivo.")
    if not isinstance(SEED, int) or isinstance(SEED, bool):
        raise ValueError("SEED debe ser entero.")
    if not isinstance(TEMPERATURE, (int, float)) or not 0 <= TEMPERATURE <= 2:
        raise ValueError("TEMPERATURE debe estar entre 0 y 2.")
    if not isinstance(MODEL_NAME, str) or not MODEL_NAME.strip():
        raise ValueError("MODEL_NAME está vacío.")
    if not isinstance(BASE_URL, str) or not BASE_URL.strip():
        raise ValueError("BASE_URL está vacío.")
    if not isinstance(API_KEY, str):
        raise ValueError("API_KEY debe ser texto.")
    if not isinstance(MAX_TOKENS, int) or MAX_TOKENS < 1:
        raise ValueError("MAX_TOKENS debe ser entero positivo.")
    if not isinstance(TIMEOUT_SECONDS, (int, float)) or TIMEOUT_SECONDS <= 0:
        raise ValueError("TIMEOUT_SECONDS debe ser positivo.")
    if PROMPT_ROLE not in {"system", "user"}:
        raise ValueError("PROMPT_ROLE debe ser 'system' o 'user'.")
    choose_targets()


def play_game(target_name, game_number):
    history = []
    constraints = empty_constraints()
    for turn in range(1, MAX_ATTEMPTS + 1):
        raw = query_model(build_messages(history))
        event = evaluate_attempt(raw, target_name, history, constraints)
        event["turn"] = turn
        history.append(event)
        show_event(event, game_number)
        if event["solved"]:
            break
    return {
        "game": game_number,
        "target": target_name,
        "solved": bool(history and history[-1]["solved"]),
        "attempts_used": len(history),
        "trace": history,
    }

## 5. Reportes y métricas

In [ ]:
def summarize(records):
    events = [event for record in records for event in record["trace"]]
    solved_records = [record for record in records if record["solved"]]
    attempts = len(events)
    invalid_count = sum(event["invalid"] for event in events)
    format_count = sum(event["format_violation"] for event in events)
    repeat_count = sum(event["repeated"] for event in events)
    violation_count = sum(event["constraint_violation"] for event in events)
    eligible_valid = sum(
        event["eligible"] and not event["invalid"] for event in events
    )
    return {
        "games": len(records),
        "solved_games": len(solved_records),
        "solve_rate": len(solved_records) / len(records) if records else 0.0,
        "attempts": attempts,
        "invalid_count": invalid_count,
        "invalid_rate": invalid_count / attempts if attempts else 0.0,
        "format_count": format_count,
        "format_rate": format_count / attempts if attempts else 0.0,
        "repeat_count": repeat_count,
        "constraint_violation_count": violation_count,
        "eligible_valid_attempts": eligible_valid,
        "constraint_violation_rate": (
            violation_count / eligible_valid if eligible_valid else None
        ),
        "average_attempts_solved": (
            sum(record["attempts_used"] for record in solved_records)
            / len(solved_records)
            if solved_records else None
        ),
    }


def violation_rows(records):
    rows = []
    for record in records:
        for event in record["trace"]:
            for violation in event["violations"]:
                rows.append({
                    "partida": record["game"],
                    "intento": event["turn"],
                    "campeón": event["guess"],
                    "atributo": LABELS[violation["attribute"]],
                    "valor usado": violation["value"],
                    "restricción previa": violation["rule"],
                })
    return rows


def attempt_rows(records):
    rows = []
    for record in records:
        for event in record["trace"]:
            guess = event.get("guess")
            if guess:
                chosen = guess
                flags = []
                if event.get("format_violation"):
                    flags.append("formato incorrecto")
                if event.get("repeated"):
                    flags.append("repetición")
                if event.get("constraint_violation"):
                    flags.append("violación de restricción")
                base_status = {
                    "solved": "resuelto",
                    "valid": "válido",
                    "repeated": "repetido",
                    "constraint_violation": "violación de restricción",
                }.get(event.get("status"), event.get("status") or "válido")
                status_parts = [base_status]
                status_parts.extend(flag for flag in flags if flag != base_status)
                status = " · ".join(status_parts)
            else:
                raw = event.get("raw_response")
                chosen = (
                    raw.strip()
                    if isinstance(raw, str) and raw.strip()
                    else "(respuesta vacía)"
                )
                status = event.get("reason") or "respuesta inválida"
            rows.append({
                "partida": record["game"],
                "intento": event.get("turn", "—"),
                "campeón elegido": chosen,
                "estado": status,
            })
    return rows


def show_report(model_name, records):
    summary = summarize(records)
    average = summary["average_attempts_solved"]
    violation_rate = summary["constraint_violation_rate"]
    print(f"\nResumen — {model_name}")
    print(f"  partidas resueltas:       {summary['solved_games']}/{summary['games']} ({summary['solve_rate']:.1%})")
    print(f"  intentos totales:         {summary['attempts']}")
    print(f"  intentos inválidos:       {summary['invalid_count']} ({summary['invalid_rate']:.1%})")
    print(f"  formato incorrecto:       {summary['format_count']} ({summary['format_rate']:.1%})")
    print(f"  repeticiones:             {summary['repeat_count']}")
    if violation_rate is None:
        print(f"  violaciones:              {summary['constraint_violation_count']} (sin intento válido posterior a feedback)")
    else:
        print(f"  violaciones:              {summary['constraint_violation_count']} ({violation_rate:.1%} de intentos válidos con feedback previo)")
    print(
        "  promedio para acertar:    "
        + (f"{average:.2f}" if average is not None else "—")
        + " intentos, solo partidas resueltas"
    )

    def render_table(title, rows, empty_message):
        if not rows:
            print(empty_message)
            return
        headers = list(rows[0])
        if display is None:
            print(f"\n{title}:")
            print(" | ".join(headers))
            for row in rows:
                print(" | ".join(str(row[key]) for key in headers))
            return
        html_rows = [
            "<tr>{}</tr>".format("".join(f"<th>{escape(str(header))}</th>" for header in headers))
        ]
        html_rows.extend(
            "<tr>{}</tr>".format(
                "".join(f"<td>{escape(str(row[header]))}</td>" for header in headers)
            )
            for row in rows
        )
        display(HTML(
            f"<h4>{escape(title)}</h4>"
            "<table style='border-collapse:collapse'>"
            + "".join(
                cell.replace("<th>", "<th style='border:1px solid #ccc;padding:4px'>")
                    .replace("<td>", "<td style='border:1px solid #ccc;padding:4px'>")
                for cell in html_rows
            )
            + "</table>"
        ))

    render_table("Detalle de intentos", attempt_rows(records), "  detalle de intentos: ninguno")
    render_table("Detalle de violaciones", violation_rows(records), "  detalle de violaciones: ninguna")


def run_experiment():
    validate_config()
    targets = choose_targets()
    print(f"Modelo: {MODEL_NAME}")
    print(f"Objetivos fijados por SEED={SEED}: {len(targets)} partidas")
    records = []
    for game_number, target_name in enumerate(targets, start=1):
        print(f"\nPartida {game_number}/{len(targets)}")
        records.append(play_game(target_name, game_number))
    show_report(MODEL_NAME, records)
    return {
        "model_name": MODEL_NAME,
        "seed": SEED,
        "targets": targets,
        "records": records,
        "summary": summarize(records),
        "violations": violation_rows(records),
    }

## 6. Self-check local

Esta celda no consulta LM Studio. Comprueba el roster, el scoring, las flechas, el parser flexible, las restricciones y los denominadores.

In [ ]:
assert len(CHAMPIONS) == 173
assert len(NAMES) == len(set(NAMES))
assert tuple(DATA_PATH.read_text(encoding="utf-8").splitlines()[0].split("|")) == EXPECTED_HEADER

same = score("Ahri", "Ahri")
assert is_solved(same) and all(status == "green" for status in same.values())

ahri_akali = score("Ahri", "Akali")
assert ahri_akali["gender"] == "green"
assert ahri_akali["positions"] == "yellow"
assert ahri_akali["species"] == "gray"
assert ahri_akali["release_year"] == "lower"
assert score("Ahri", "Aatrox")["release_year"] == "higher"

assert parse_response("Ahri") == {
    "name": "Ahri", "format_ok": True, "reason": "",
}
assert parse_response("  kAi sa  ")["name"] == "Kai'Sa"
assert parse_response("Creo que es Kai'Sa.")["name"] == "Kai'Sa"
assert not parse_response("Creo que es Kai'Sa.")["format_ok"]
assert parse_response("Ahri y Akali")["name"] is None
assert parse_response("Gandalf")["name"] is None
assert parse_response('{"guess":"Ahri"}')["name"] == "Ahri"

_original_http_json = _http_json
try:
    captured_requests = []
    def _valid_http_json(path, body):
        captured_requests.append((path, body))
        return {"choices": [{"finish_reason": "stop", "message": {"content": "Ahri"}}]}
    _http_json = _valid_http_json
    assert query_model([]) == "Ahri"
    assert "stop" not in captured_requests[0][1]

    def _empty_http_json(path, body):
        return {"choices": [{"finish_reason": "length", "message": {"content": "", "reasoning_content": "Thinking..."}}]}
    _http_json = _empty_http_json
    try:
        query_model([])
    except RuntimeError as exc:
        assert "reasoning_content" in str(exc)
    else:
        raise AssertionError("Una respuesta vacía con razonamiento debía fallar")
finally:
    _http_json = _original_http_json

format_attempt = evaluate_attempt("Creo que es Kai'Sa.", "Garen", [], empty_constraints())
format_attempt["turn"] = 1
invalid_attempt = evaluate_attempt("Gandalf", "Garen", [], empty_constraints())
invalid_attempt["turn"] = 2
ambiguous_attempt = evaluate_attempt("Ahri y Akali", "Garen", [], empty_constraints())
ambiguous_attempt["turn"] = 3
empty_attempt = evaluate_attempt("", "Garen", [], empty_constraints())
empty_attempt["turn"] = 4
attempt_test_rows = attempt_rows([{"game": 1, "trace": [format_attempt, invalid_attempt, ambiguous_attempt, empty_attempt]}])
assert attempt_test_rows[0]["campeón elegido"] == "Kai'Sa"
assert "formato incorrecto" in attempt_test_rows[0]["estado"]
assert attempt_test_rows[1]["campeón elegido"] == "Gandalf"
assert "roster" in attempt_test_rows[1]["estado"]
assert attempt_test_rows[2]["campeón elegido"] == "Ahri y Akali"
assert "más de un campeón" in attempt_test_rows[2]["estado"]
assert attempt_test_rows[3]["campeón elegido"] == "(respuesta vacía)"

lux_constraints = empty_constraints()
lux_attempt = evaluate_attempt("Lux", "Garen", [], lux_constraints)
lux_attempt["turn"] = 1
invalid_after_lux = evaluate_attempt("Gandalf", "Garen", [lux_attempt], lux_constraints)
invalid_after_lux["turn"] = 2
lux_rows = attempt_rows([{"game": 2, "trace": [lux_attempt, invalid_after_lux]}])
assert lux_rows[1]["campeón elegido"] == "Gandalf"

constraints = empty_constraints()
first = evaluate_attempt("Ahri", "Garen", [], constraints)
assert first["feedback"]["gender"] == "gray"
second = evaluate_attempt("Lux", "Garen", [first], constraints)
assert second["constraint_violation"]
assert any(item["attribute"] == "gender" for item in second["violations"])

green_constraints = empty_constraints()
green_first = evaluate_attempt("Akali", "Ahri", [], green_constraints)
green_second = evaluate_attempt("Garen", "Ahri", [green_first], green_constraints)
assert any(item["attribute"] == "gender" for item in green_second["violations"])

year_constraints = empty_constraints()
year_feedback = {field: "yellow" for field in FEEDBACK_FIELDS}
year_feedback["release_year"] = "higher"
update_constraints(year_constraints, "Aatrox", year_feedback)
assert any(
    item["attribute"] == "release_year"
    for item in constraint_violations("Aatrox", year_constraints)
)

repeat_constraints = empty_constraints()
repeat_first = evaluate_attempt("Ahri", "Garen", [], repeat_constraints)
repeat_second = evaluate_attempt("Ahri", "Garen", [repeat_first], repeat_constraints)
assert repeat_second["repeated"] and repeat_second["constraint_violation"]

assert choose_targets() == choose_targets()
assert len(choose_targets()) == len(set(choose_targets()))

fake_records = [{
    "game": 1,
    "solved": True,
    "attempts_used": 2,
    "trace": [
        {
            "invalid": True, "format_violation": True, "repeated": False,
            "constraint_violation": False, "eligible": False, "feedback": None,
            "violations": [],
        },
        {
            "invalid": False, "format_violation": True, "repeated": True,
            "constraint_violation": True, "eligible": True, "feedback": {},
            "violations": [{"attribute": "gender", "value": "Female", "rule": "gris"}],
        },
    ],
}]
fake_summary = summarize(fake_records)
assert fake_summary["attempts"] == 2
assert fake_summary["invalid_count"] == 1
assert fake_summary["format_count"] == 2
assert fake_summary["repeat_count"] == 1
assert fake_summary["constraint_violation_count"] == 1
assert fake_summary["eligible_valid_attempts"] == 1
assert fake_summary["constraint_violation_rate"] == 1.0
assert fake_summary["average_attempts_solved"] == 2

print("Self-check OK: roster, feedback, flechas, parser, cliente LM Studio, restricciones y métricas.")

## 7. Ejecutar

Los errores de conexión no se convierten en intentos inválidos. Corrige BASE_URL, API_KEY o MODEL_NAME y vuelve a ejecutar la celda final.

In [ ]:
if RUN_NOW:
    results = run_experiment()
else:
    print("RUN_NOW=False: self-check listo; define MODEL_NAME y cambia RUN_NOW=True para jugar.")